# Customer Feature Reductions

This notebook loads the engineered customer feature matrix and compares PCA against a VAE latent space.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC_DIR = ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from embeddings.customer_features import CustomerFeatureConfig, build_customer_feature_matrix, read_lumen_csv
from embeddings.reductions import (
    TORCH_AVAILABLE,
    RecommendationAwareVAEConfig,
    build_recommendation_targets,
    evaluate_latent_space,
    fit_pca_embedding,
    preprocess_feature_matrix,
    train_recommendation_aware_vae,
    train_vae,
)

DATA_PATH = Path(r'C:\\Users\\lovro\\Desktop\\hackatoni\\LUMEN_DS_processed.csv')
pd.set_option('display.max_columns', 200)


In [ ]:
raw_df = read_lumen_csv(DATA_PATH)
customer_features = build_customer_feature_matrix(raw_df, CustomerFeatureConfig(bucket_size=5))
feature_df, X, imputer, scaler = preprocess_feature_matrix(customer_features)
targets = build_recommendation_targets(feature_df)
print('feature frame shape:', feature_df.shape)
print('matrix shape:', X.shape)
print('auxiliary target shapes:', {name: value.shape for name, value in targets.items()})
print('torch available:', TORCH_AVAILABLE)


In [ ]:
pca_result = fit_pca_embedding(X, n_components=16)
print('PCA explained variance:', round(pca_result.explained_variance_ratio_sum, 4))
display(evaluate_latent_space(pca_result.latent))
display(pd.DataFrame(pca_result.latent[:5], index=feature_df.index[:5]))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(pca_result.model.explained_variance_ratio_) + 1), pca_result.model.explained_variance_ratio_.cumsum())
plt.title('PCA Cumulative Explained Variance')
plt.xlabel('Component')
plt.ylabel('Cumulative Explained Variance')
plt.tight_layout()
plt.show()


## Recommendation-Aware VAE Loss Variants

The three supervised variants below are:

- `product_group`: reconstruct features and also predict the per-customer `Product group` share vector
- `bucketed`: reconstruct features and also predict bucket-local product-group shares plus bucket-level count/revenue magnitudes
- `hybrid`: use both `Product group` and bucket-aware auxiliary targets


In [ ]:
if not TORCH_AVAILABLE:
    raise ImportError('torch is not installed in classic_methods/.venv. Install dependencies and rerun this notebook.')

plain_vae_result = train_vae(X, latent_dim=16, beta=0.05, epochs=80)
product_group_result = train_recommendation_aware_vae(
    X,
    feature_df=feature_df,
    config=RecommendationAwareVAEConfig(loss_variant='product_group', latent_dim=16, beta=0.05, auxiliary_weight=2.0, epochs=80),
)
bucketed_result = train_recommendation_aware_vae(
    X,
    feature_df=feature_df,
    config=RecommendationAwareVAEConfig(loss_variant='bucketed', latent_dim=16, beta=0.05, auxiliary_weight=1.5, bucket_share_weight=1.0, bucket_magnitude_weight=0.5, epochs=80),
)
hybrid_result = train_recommendation_aware_vae(
    X,
    feature_df=feature_df,
    config=RecommendationAwareVAEConfig(loss_variant='hybrid', latent_dim=16, beta=0.05, auxiliary_weight=1.5, bucket_share_weight=1.0, bucket_magnitude_weight=0.5, epochs=80),
)

for name, result in {
    'plain': plain_vae_result,
    'product_group': product_group_result,
    'bucketed': bucketed_result,
    'hybrid': hybrid_result,
}.items():
    print(name)
    display(result.history.tail())
    display(evaluate_latent_space(result.latent_mean))
    display(pd.DataFrame(result.latent_mean[:5], index=feature_df.index[:5]))

plt.figure(figsize=(10, 5))
plt.plot(plain_vae_result.history['epoch'], plain_vae_result.history['loss_per_row'], label='plain total')
plt.plot(product_group_result.history['epoch'], product_group_result.history['loss_per_row'], label='product_group total')
plt.plot(bucketed_result.history['epoch'], bucketed_result.history['loss_per_row'], label='bucketed total')
plt.plot(hybrid_result.history['epoch'], hybrid_result.history['loss_per_row'], label='hybrid total')
plt.title('VAE Variant Training Curves')
plt.xlabel('Epoch')
plt.ylabel('Loss per Row')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
comparison = pd.DataFrame(index=feature_df.index)
comparison['pca_norm'] = (pca_result.latent ** 2).sum(axis=1) ** 0.5
comparison['row_count'] = feature_df['activity__row_count']
comparison['revenue_sum'] = feature_df.get('agg__Invoiced price__sum', 0.0)
if TORCH_AVAILABLE:
    comparison['plain_vae_norm'] = (plain_vae_result.latent_mean ** 2).sum(axis=1) ** 0.5
    comparison['product_group_vae_norm'] = (product_group_result.latent_mean ** 2).sum(axis=1) ** 0.5
    comparison['bucketed_vae_norm'] = (bucketed_result.latent_mean ** 2).sum(axis=1) ** 0.5
    comparison['hybrid_vae_norm'] = (hybrid_result.latent_mean ** 2).sum(axis=1) ** 0.5
display(comparison.describe().T)
display(comparison.sort_values('revenue_sum', ascending=False).head(20))
